In [1]:
import os
from itertools import chain

from dotenv import load_dotenv
from langchain_core import messages
from langchain_core.messages import HumanMessage
from langchain_protocol import Command

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph_examples.checkpoint.memory import InMemorySaver


def read_email_tool(email: str) -> str:
    """ Mock function to read email """
    return f"{email}"


def send_email_tool(email: str, subject: str, body: str) -> str:
    """ Mock function to read email """
    return f"Email sent to {email} {subject}"


config = {"configurable": {"thread_id": "test-approve"}}

agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_email_tool": False
            }
        ),

    ]
)

result = agent.invoke({"messages": [HumanMessage(content="Send email to abc@gmail.com with subject and Body as Test message")]},
                      config=config)
result


C:\Users\Navoki\miniconda3\envs\langchain_practice\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'messages': [HumanMessage(content='Send email to abc@gmail.com with subject and Body as Test message', additional_kwargs={}, response_metadata={}, id='bf03549a-e4ef-4e7c-8733-a53ab50af288'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 86, 'total_tokens': 113, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_ee9850c3c6', 'id': 'chatcmpl-Dga0nyaRAmGESH22FpPVz751A0GAm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3714-e2b1-7f51-b423-6105ade31b51-0', tool_calls=[{'name': 'send_email_tool', 'args': {'email': 'abc@gmail.com', 'subject': 'Test message', 'body': 'Test message'}, 'id': 'call_iT8unnKrD4xKGLxC3JTN

In [2]:
from langgraph_examples.types import Command
if "__interrupt__" in result:
    print("Pause Approving...")

    result = agent.invoke(Command(resume={
        "decisions":[
            {"type":"approve"}
        ]
    }),
    config=config)


print(f"Approved: {result["messages"][-1].content}")

Pause Approving...
Approved: The email has been sent to abc@gmail.com with the subject and body as "Test message."
